# Access Requests with the ML App Python client

A focused, rerunnable walkthrough of Business Case discovery, auditable access requests and bounded request queues. It does not expose inaccessible Business Case details.


In [ ]:
from pathlib import Path
import sys

repository_root = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'ml_app_client').is_dir()), None)
if repository_root is None:
    raise RuntimeError('Start Jupyter inside the ml-app repository')
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from ml_app_client import MLAppClient

client = MLAppClient.connect()
print('Connected as', client.me().get('login_name'))


## Find a Business Case without reading its details

The organization-wide directory is a bounded, minimal projection. Select an entry before requesting access; do not infer case details from the directory.


In [ ]:
SEARCH = 'churn'
directory = client.page_business_case_catalog(search=SEARCH, limit=30)
print(f'{directory.total} matching directory entries')
[(entry.id, entry.name, entry.access_role, entry.request_status) for entry in directory.items]


## Submit a request deliberately

Uncomment this only after choosing an entry you do not already have access to. Explain the work that needs the requested role. `owner` is never requestable; ownership transfers use the separate audited Business Case workflow.


In [ ]:
# entry = directory.items[0]
# request = client.request_business_case_access(
#     entry.id,
#     requested_role='reader',
#     justification='I maintain the monthly churn report.',
# )
# print(request.id, request.status)


## Inspect only the queue you need

`incoming` is for cases you can manage; `mine` is your submitted queue. Both are bounded pages. Use `submitted_history` or `handled` when reviewing completed activity.


In [ ]:
submitted = client.page_business_case_access_requests(box='mine', limit=30)
incoming = client.page_business_case_access_requests(box='incoming', limit=30)
print({'submitted_pending': submitted.total, 'incoming_pending': incoming.total})

# For one manageable case, use a focused queue or its decision history:
# case_requests = client.page_business_case_access_requests_for_business_case(
#     business_case_id, history=False, limit=30
# )


## Decide an incoming request

A manager can approve a pending request and grant an allowed role atomically, or reject it with a recorded note. The selected role cannot exceed the manager's effective role.


In [ ]:
# request = incoming.items[0]
# approved = client.approve_business_case_access_request(
#     request.id, access_role='reader', decision_note='Approved for reporting.',
# )
# rejected = client.reject_business_case_access_request(
#     request.id, decision_note='Access is not required for this work.',
# )
# print(approved.status, rejected.status)
